# 02 · Model training (Random Forest + XGBoost)

Trains the two models with the same hyperparameters as the ASTRA API artifacts and saves
`rf_multiclass.pkl` and `xgb_multiclass.pkl` next to the scaler and label encoder from notebook 01.
No resampling (no SMOTE) is used; class imbalance is handled by `class_weight="balanced"` in the Random Forest.

**Needs:** output of notebook 01. **Produces:** the four files that go into the repo's `artifacts/` folder.

In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

WORK_DIR = Path(os.environ.get("ASTRA_WORK_DIR", "astra_run"))   # shared by all four notebooks
ART_DIR = WORK_DIR / "artifacts"
SEED = 42
CLASSES = ["benign", "bruteforce", "ddos", "dos", "portscan"]   # alphabetical = LabelEncoder order
ART_DIR.mkdir(parents=True, exist_ok=True)

import sklearn, xgboost, platform
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

N_JOBS = -1
S = joblib.load(WORK_DIR / "splits.joblib")
le = joblib.load(ART_DIR / "label_encoder.pkl")
meta = json.loads((WORK_DIR / "data_meta.json").read_text())
X_train, y_train = S["X_train"], S["y_train"]
print("training matrix:", X_train.shape)

## Random Forest

In [ ]:
RF_PARAMS = dict(n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=N_JOBS)
t = time.time()
rf = RandomForestClassifier(**RF_PARAMS).fit(X_train, y_train)
rf_time = time.time() - t
print(f"Random Forest trained in {rf_time/60:.1f} min")

## XGBoost

In [ ]:
XGB_PARAMS = dict(objective="multi:softprob", n_estimators=100, learning_rate=0.1, max_depth=6,
                  subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                  eval_metric="mlogloss", random_state=SEED, n_jobs=N_JOBS)
t = time.time()
xgb = XGBClassifier(**XGB_PARAMS).fit(X_train, y_train)
xgb_time = time.time() - t
print(f"XGBoost trained in {xgb_time/60:.1f} min")

## Feature importance (sanity check)

In [ ]:
feats = np.array(meta["features"])
imp = pd.DataFrame({"random_forest": rf.feature_importances_, "xgboost": xgb.feature_importances_}, index=feats)
top = imp.assign(mean=imp.mean(axis=1)).sort_values("mean", ascending=False).head(15)[["random_forest", "xgboost"]]
ax = top.iloc[::-1].plot.barh(figsize=(7, 5)); ax.set_title("Top-15 features (impurity / gain importance)")
plt.tight_layout(); plt.show()

## Save artifacts

In [ ]:
joblib.dump(rf, ART_DIR / "rf_multiclass.pkl")
joblib.dump(xgb, ART_DIR / "xgb_multiclass.pkl")
train_meta = {"rf_params": RF_PARAMS, "xgb_params": XGB_PARAMS,
              "train_seconds": {"rf": round(rf_time, 1), "xgb": round(xgb_time, 1)},
              "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                           "xgboost": xgboost.__version__, "numpy": np.__version__, "pandas": pd.__version__}}
(WORK_DIR / "training_meta.json").write_text(json.dumps(train_meta, indent=2), encoding="utf-8")
print(sorted(p.name for p in ART_DIR.iterdir()))
print(train_meta["versions"])
print("Copy the four .pkl files to the repo's artifacts/ folder and pin the same sklearn/xgboost versions in requirements.txt.")